# FASE 1. COMPRENSIÓN DE LOS DATOS - ESPIRAS Y COORDENADAS

## 1. Seteo de la Sesion Spark

Para poder linkear las espiras con los distritos, necesitamos arreglar algunos de los problemas detectados durante el análisis de trafico

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window

spark = SparkSession.builder \
    .appName("TFM_Espiras_Ubicaciones_Coordenadas") \
    .getOrCreate()

:: loading settings :: url = jar:file:/usr/lib/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b4e4c120-4146-4aa0-b3d3-fa4daf799985;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.3.2 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.3.2 in central
	found org.apache.kafka#kafka-clients;2.8.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.8.4 in central
	found org.slf4j#slf4j-api;1.7.32 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.2 in central
	found org.spark-project.spark#unused;1.0.0 in central
	found org.apache.hadoop#hadoop-client-api;3.3.2 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report 

# 2. Análisis de espiras 

Del total de 965 espiras finales, analizaremos cuáles tienen coordenadas válidas y cuáles no.
El objetivo es que cada espira tenga coordenadas válidas que nos permitan relacionarlas con el distrito correspondiente.

### 2.1 Espiras únicas del archivo final de trafico (después de la limpieza)

In [3]:
espiras_limpias = (spark.read.parquet("gs://tfm_unir1/processed/trafico/espiras/")
            .select("id_espira").distinct())

print("Espiras limpias:", espiras_limpias.count())

Espiras limpias: 965


### 2.2 Ubicaciones de cada espira en los datos originales

In [4]:
archivo_trafico_inicial = spark.read.parquet("gs://tfm_unir1/trafico/espires_VLC2_unificado")

ubicaciones = (archivo_trafico_inicial
    .select(F.col("detector_id").alias("id_espira"), "latitude", "longitude", "snapshot_datetime")
    .join(espiras_limpias, "id_espira")
    .groupBy("id_espira", "latitude", "longitude")
    .agg(
        F.count("*").alias("registros"),
        F.min("snapshot_datetime").alias("desde"),
        F.max("snapshot_datetime").alias("hasta")))

ubicaciones.cache()
print("Combinaciones espira-ubicación:", ubicaciones.count())

Combinaciones espira-ubicación: 1453


Tenemos 1453 combinaciones. Es decir que hay espiras que tienen mas de una ubicación. Analizaremos caso por caso.

### 2.3 Espiras con latitud y longitud dentro de Valencia

In [5]:
ubicaciones = ubicaciones.withColumn("valida",
    F.coalesce(
        F.col("latitude").between(39.27, 39.57) & F.col("longitude").between(-0.45, -0.27),
        F.lit(False)))

ubicaciones.groupBy("valida").count().show()

+------+-----+
|valida|count|
+------+-----+
|  true| 1435|
| false|   18|
+------+-----+



De las 1453 tenemos 18 con ubicacion fuera de Valencia

### 2.4 Cuántas ubicaciones tiene cada espira

In [6]:
resumen = ubicaciones.groupBy("id_espira").agg(
    F.count("*").alias("n_ubicaciones"),
    F.sum(F.col("valida").cast("int")).alias("n_validas"))

resumen.groupBy("n_ubicaciones", "n_validas").count() \
    .orderBy("n_ubicaciones", "n_validas").show()

+-------------+---------+-----+
|n_ubicaciones|n_validas|count|
+-------------+---------+-----+
|            1|        1|  492|
|            2|        1|   10|
|            2|        2|  448|
|            3|        2|    8|
|            3|        3|    7|
+-------------+---------+-----+



**Observaciones:**
- De las 965 espiras, 492 tienen una única ubicación y es válida.
- 473 espiras tienen más de una ubicación:
    - 455 tienen 2 o 3 ubicaciones y todas son válidas (448 con dos y 7 con tres).
    - 18 tienen alguna ubicación inválida (fuera de Valencia o en 0,0), pero todas tienen además al menos una ubicación válida (10 con una válida y 8 con dos válidas).
- Ninguna espira se queda sin una ubicación válida.

Analizamos en los pasos siguientes cuándo aparece cada ubicación y cuánto se separan.

### 2.5 Qué tan alejadas estan las ubicaciones válidas de una misma espira

In [13]:
separacion = (ubicaciones
    .filter(F.col("valida"))
    .groupBy("id_espira")
    .agg(
        F.count("*").alias("n_validas"),
        F.round((F.max("latitude") - F.min("latitude")) * 111000).alias("metros_norte_sur"),
        F.round((F.max("longitude") - F.min("longitude")) * 86000).alias("metros_este_oeste"))
    .filter(F.col("n_validas") > 1)
    # Distancia total: combina norte-sur y este-oeste (Pitágoras)
    .withColumn("metros_total",
        F.round(F.sqrt(F.col("metros_norte_sur")**2 + F.col("metros_este_oeste")**2)))
    .orderBy(F.desc("metros_total")))

separacion.show(20)

+---------+---------+----------------+-----------------+------------+
|id_espira|n_validas|metros_norte_sur|metros_este_oeste|metros_total|
+---------+---------+----------------+-----------------+------------+
|      829|        2|           160.0|            100.0|       189.0|
|      911|        2|            60.0|              9.0|        61.0|
|     3928|        2|            19.0|             53.0|        56.0|
|     4107|        3|            53.0|             16.0|        55.0|
|     3031|        3|             5.0|             52.0|        52.0|
|     2002|        2|            13.0|             36.0|        38.0|
|     4411|        2|            10.0|             36.0|        37.0|
|     2317|        2|            36.0|              4.0|        36.0|
|     1523|        2|            33.0|             13.0|        35.0|
|     2538|        2|            17.0|             29.0|        34.0|
|     2525|        2|            19.0|             28.0|        34.0|
|     2412|        2

In [14]:
separacion = (ubicaciones
    .filter(F.col("valida"))
    .groupBy("id_espira")
    .agg(
        F.count("*").alias("n_validas"),
        F.round((F.max("latitude") - F.min("latitude")) * 111000).alias("metros_norte_sur"),
        F.round((F.max("longitude") - F.min("longitude")) * 86000).alias("metros_este_oeste"))
    .filter(F.col("n_validas") > 1)
    # Distancia total: combina norte-sur y este-oeste (Pitágoras)
    .withColumn("metros_total",
        F.round(F.sqrt(F.col("metros_norte_sur")**2 + F.col("metros_este_oeste")**2)))
    .orderBy(F.asc("metros_total")))

separacion.show(20)

+---------+---------+----------------+-----------------+------------+
|id_espira|n_validas|metros_norte_sur|metros_este_oeste|metros_total|
+---------+---------+----------------+-----------------+------------+
|     1107|        2|             0.0|              0.0|         0.0|
|      603|        2|             0.0|              0.0|         0.0|
|      412|        2|             0.0|              0.0|         0.0|
|      511|        2|             0.0|              0.0|         0.0|
|     3324|        2|             0.0|              0.0|         0.0|
|     2624|        2|             1.0|              1.0|         1.0|
|     2326|        2|             1.0|              0.0|         1.0|
|      519|        2|             1.0|              1.0|         1.0|
|      709|        2|             1.0|              0.0|         1.0|
|     2506|        2|             1.0|              1.0|         1.0|
|     2324|        2|             1.0|              1.0|         1.0|
|      402|        2

Algunas estan muy separadas (829 --> 189 m)pero la mayoria esta a una distancia de aprox 60 m, lo que podría significar solo un cambio de referenciacion

In [18]:
ubicaciones.filter(F.col("id_espira").isin("1001", "101", "116", "829", "1531", "911")) \
    .orderBy("id_espira", F.desc("registros")).show(truncate=False)

+---------+------------------+-------------------+---------+-------------------+-------------------+------+
|id_espira|latitude          |longitude          |registros|desde              |hasta              |valida|
+---------+------------------+-------------------+---------+-------------------+-------------------+------+
|1001     |39.48020371479589 |-0.3921767308038595|63873    |2023-10-16 03:15:01|2025-09-24 00:00:02|true  |
|1001     |39.48030570110385 |-0.3922689548662662|19915    |2023-03-09 17:00:02|2023-10-16 03:00:02|true  |
|101      |39.47058364869547 |-0.3738167430612965|63844    |2023-10-16 03:15:01|2025-09-24 00:00:02|true  |
|101      |39.47060777618564 |-0.3737717823813373|19897    |2023-03-09 17:00:02|2023-10-16 03:00:02|true  |
|116      |39.471691841228505|-0.3742825293478465|63877    |2023-10-16 03:15:01|2025-09-24 00:00:02|true  |
|116      |-7.488743884877489|0.0                |19917    |2023-03-09 17:00:02|2023-10-16 03:00:02|false |
|1531     |39.47208005929855

In [19]:
# Espiras con más de una ubicación
varias = resumen.filter(F.col("n_ubicaciones") > 1).select("id_espira")

# De esas, cuántas tienen una ubicación que empieza el 2023-10-16
cambio = (ubicaciones
    .join(varias, "id_espira")
    .filter(F.to_date("desde") == "2023-10-16")
    .select("id_espira").distinct())

print("Espiras con más de una ubicación:", varias.count())
print("Espiras con cambio de ubicación el 2023-10-16:", cambio.count())

Espiras con más de una ubicación: 473


Espiras con cambio de ubicación el 2023-10-16: 469


**Observaciones:**
- Se detecta un patrón en las que tienen mas de una ubicacion: el cambio de ubicacion se produce el 16 de Octubre de 2023.
- Las dos con valores invalidos, parecieran tener valores válidos modificados en esa misma fecha (dejan de tener latitud negativa)
- La 829 tiene dos dos valores a la vez cuya distancia es 189. 

### 2.6 Conclusiones del análisis

- De las 965 espiras, 492 tienen una única ubicación y es válida.
- 18 combinaciones de 1.453 son inválidas (fuera de València o en 0,0). Todas pertenecen a espiras que tienen además una ubicación válida: ninguna espira se queda sin coordenadas.
- 463 espiras tienen 2 o 3 ubicaciones válidas. El **16 de octubre de 2023** la fuente actualizó la georreferenciación: hasta esa fecha se usan unas coordenadas y desde entonces otras. Las diferencias son en general de menos de 20 metros (máximo unos 60 m). La actualización también corrigió las coordenadas inválidas (ejemplo: espira 116).
- La espira **829** es un caso distinto: desde el 1 de octubre de 2024 aparecen dos ubicaciones **a la vez**, a unos 190 m. El mismo identificador corresponde a dos detectores, por lo que sus mediciones no se pueden atribuir a un único punto.